# MAPNAI — Bespoke-MiniCheck-7B on Colab (S5 ceiling, optional)

This is the **optional, A100-only** part that used to be Part 3 of `mapnai_gpu_runs.ipynb`.
It scores the LLM-AggreFact **test** split (29,320 pairs) with Bespoke-MiniCheck-7B (published 77.4) — about 1 h on an A100.

**Needs:** an **A100 (40 or 80 GB)** runtime (Colab Pro / compute units) and the same Hugging Face **read** token as the main notebook.
It is independent of `mapnai_gpu_runs.ipynb`: run it in a fresh session, before or after the main notebook.
It writes to the same Drive folder (`My Drive/mapnai_colab`) with the same tag, so the results zip contains everything scored so far.

Scores are saved row by row to Google Drive: if Colab disconnects, reconnect and run every cell again from the top — finished rows are skipped.

The model licence is CC BY-NC 4.0 (non-commercial). It needs `trust_remote_code`: cell 1.1 checks that its code files are
exactly the ones reviewed on 2026-10-23 before anything is loaded.

**Order:** run every cell top to bottom.

## Part 0 — Setup (run every cell, once per session)

In [ ]:
# 0.1  Which GPU did Colab give you?  (Runtime > Change runtime type > A100 GPU)
!nvidia-smi --query-gpu=name,memory.total --format=csv
import subprocess
mem_mib = int(subprocess.run(['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
                             capture_output=True, text=True).stdout.split()[0])
assert mem_mib >= 38000, f'Only {mem_mib} MiB of GPU memory: this notebook needs an A100 (40/80 GB). Change the runtime type.'
print('GPU OK')

In [ ]:
# 0.2  Google Drive keeps the scores safe across disconnects. Approve the pop-up.
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/mapnai_colab'          # everything is written here
TAG = 'colab-2026-10-24'                              # do not change between sessions: it is how rows are resumed
CKPTS = '/content/ckpts'                              # model files: Colab's local disk (Drive cannot hold the HF cache's symlinks)
for d in ('data', 'cache', 'outputs'):
    os.makedirs(f'{WORK}/{d}', exist_ok=True)
print('working folder:', WORK)

In [ ]:
# 0.3  vLLM and minicheck's LLM extra (several minutes). If Colab asks to restart, click "Restart session",
#      then run this notebook again from 0.1 (this cell will finish quickly the second time).
!pip install -q "minicheck[llm] @ git+https://github.com/Liyan06/MiniCheck.git@b58b9fa69acbd1015ec970fa65dd752413a053d2" "datasets==3.6.0" "scikit-learn==1.7.2" "nltk==3.9.1"
import nltk
nltk.download('punkt_tab', quiet=True)
os.makedirs('/root/nltk_data/tokenizers/punkt', exist_ok=True)

In [ ]:
# 0.4  Log in to Hugging Face: paste your READ token when asked (it is not saved in this notebook).
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 0.5  Download the pinned dataset (skipped if already on Drive) and verify it by sha256.
import hashlib
from huggingface_hub import snapshot_download, HfApi

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 22), b''):
            h.update(b)
    return h.hexdigest()

DATA_REV = '981dfd0bd8e58e7238a9ab92b2e6ea44bce918e4'
data_dir = snapshot_download('lytang/LLM-AggreFact', repo_type='dataset', revision=DATA_REV,
                             allow_patterns=['data/*.parquet'], local_dir=f'{WORK}/data')
EXPECTED = {'data/test-00000-of-00001.parquet': 'ba6976c3da990a9d7b11670efaf4e3c93c1bd0530061ec742afd6b2d5bce698c',
            'data/dev-00000-of-00001.parquet': 'ca9a5bda9561bbc6f7327ef81ab41727fa10e96f96e9dc74ef32ba3af3c1d8a8'}
for fn, want in EXPECTED.items():
    assert sha256(f'{data_dir}/{fn}') == want, f'{fn} does not match the pinned dataset'
print('dataset OK')

In [ ]:
# 0.6  The summary code (same protocol as research/experiments/e20_aggrefact_eval.py).
import json, time
import numpy as np
import pandas as pd

def load_split(split):
    return pd.read_parquet(f'{WORK}/data/data/{split}-00000-of-00001.parquet').reset_index(drop=True)

def cache_path(checker, split):
    return f'{WORK}/cache/e20_{checker}_{split}_{TAG}.jsonl'

def bacc(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return float(np.nanmean([(p[y == 1] == 1).mean(), (p[y == 0] == 0).mean()]) * 100)

def summary(checker, split):
    df = load_split(split)
    probs = {}
    for l in open(cache_path(checker, split), encoding='utf-8'):
        r = json.loads(l)
        probs[r['row']] = r['prob']
    missing = len(df) - len(probs)
    df = df[df.index.isin(probs)].copy()
    df['pred'] = [int(probs[i] > 0.5) for i in df.index]
    per = {ds: round(bacc(g.label, g.pred), 1) for ds, g in df.groupby('dataset')}
    avg = round(sum(per.values()) / len(per), 2)
    news3 = [per[d] for d in ('AggreFact-CNN', 'AggreFact-XSum', 'TofuEval-MediaS') if d in per]
    print(f'{checker} {split}: {len(df)} rows scored, {missing} missing')
    print(per)
    print('11-dataset average:', avg, '| News-3 mean:', round(sum(news3) / len(news3), 2) if news3 else None)
    if missing == 0 and split == 'test' and checker == 'minicheck-ft5':
        print('S1 gap vs the published 75.0:', round(avg - 75.0, 2), '(pass if within +/-2.0)')

print('ready')

## Part 1 — Score the test split with Bespoke-MiniCheck-7B

In [ ]:
# 1.1  Pinned model + check of the reviewed remote code.
BESPOKE, BESPOKE_REV = 'bespokelabs/Bespoke-MiniCheck-7B', '1ed7786bcda3fa1dc35f7c4ed9e3f36b785d33b8'
CODE_SHA256 = {'configuration_internlm2.py': '89863f76efbb7f31b39e8feb789285b7362d243e7c9c2a07ac018e0e313fd856',
               'modeling_internlm2.py': '1b781ec759a151caa48e03641e9b01411376f157cacd9a28044a714e4abad005',
               'tokenization_internlm2.py': '444d4c2b0da158e61c34b3c727943f0ad454770c74b307f4d881f03603335eef',
               'tokenization_internlm2_fast.py': '1fe5f033032afc400b9b8b0274f16efd57c6ebaab46a2f6fe9f961afb868ae10'}
assert HfApi().model_info(BESPOKE).sha == BESPOKE_REV, 'Bespoke main moved: stop and tell Claude'
bpath = snapshot_download(BESPOKE, revision=BESPOKE_REV, cache_dir=CKPTS)
for fn, want in CODE_SHA256.items():
    assert sha256(f'{bpath}/{fn}') == want, f'{fn} changed since the review: stop and tell Claude'
print('Bespoke-MiniCheck-7B OK')

In [ ]:
# 1.2  Score the test split in chunks of 2,000 rows (resumable).
from minicheck.minicheck import MiniCheck
df, path = load_split('test'), cache_path('bespoke-7b', 'test')
done = {json.loads(l)['row'] for l in open(path, encoding='utf-8')} if os.path.exists(path) else set()
todo = [i for i in range(len(df)) if i not in done]
print(len(done), 'done,', len(todo), 'to score')
if todo:
    scorer = MiniCheck(model_name='Bespoke-MiniCheck-7B', enable_prefix_caching=False, cache_dir=CKPTS)
    t0 = time.time()
    with open(path, 'a', encoding='utf-8') as fh:
        for s in range(0, len(todo), 2000):
            ids = todo[s:s + 2000]
            _, probs, _, _ = scorer.score(docs=[df.doc[i] for i in ids], claims=[df.claim[i] for i in ids])
            for i, p in zip(ids, probs):
                fh.write(json.dumps({'row': i, 'prob': float(p)}) + '\n')
            fh.flush()
            print(f'  {s + len(ids)}/{len(todo)}  {(time.time() - t0) / 60:.0f} min', flush=True)
summary('bespoke-7b', 'test')

## Part 2 — Package the results (run at the end of every session that produced scores)

In [ ]:
import platform, subprocess, zipfile
import torch
env = {'tag': TAG, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       'python': platform.python_version(),
       'pip_freeze': subprocess.run(['pip', 'freeze'], capture_output=True, text=True).stdout.splitlines(),
       'cache_files': {f: sum(1 for _ in open(f'{WORK}/cache/{f}', encoding='utf-8'))
                       for f in sorted(os.listdir(f'{WORK}/cache'))}}
gpu_tag = (env['gpu'] or 'cpu').split()[-1]
json.dump(env, open(f'{WORK}/outputs/environment_{TAG}_{gpu_tag}.json', 'w'), indent=1)
zip_path = f'{WORK}/mapnai_colab_results_{TAG}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for d in ('cache', 'outputs'):
        for f in os.listdir(f'{WORK}/{d}'):
            z.write(f'{WORK}/{d}/{f}', f'{d}/{f}')
print(json.dumps(env['cache_files'], indent=1))
print('Download this file from Google Drive:', zip_path)